# core

> Fill in a module description here

In [1]:
# | default_exp gradio_app

In [2]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [ ]:
import gradio as gr
from product_horse.api import (
    create_user_and_add_files,
    transcribe_file_and_extract_speakers,
    create_and_save_schema,
    extract_info_from_transcriptions,
    FileDict,
)
from product_horse.db import SqlModelDatabase, UnvalidatedSchema
from product_horse.audio_video import AudioEditor
from product_horse.filesystems import LocalFileSystem
from product_horse.core import InMemoryQueue
from product_horse.extraction import AIModelClient
import os

# put the app working directory at ~/.product_horse/
app_working_directory = os.path.expanduser("~/.product_horse/")

db = SqlModelDatabase(database_url=app_working_directory + "product_horse.db")
fs = LocalFileSystem(base_path=app_working_directory + "files/")
ai_model_client = AIModelClient()
audio_editor = AudioEditor()
queue = InMemoryQueue()


def save_files_and_transcriptions(user_id: str, files: list):
    user_data = {"id": user_id}
    file_dicts: list[FileDict] = [
        {"content": file.read(), "name": file.name} for file in files
    ]
    _, metadata = create_user_and_add_files(user_data, file_dicts, db, fs)
    for file_metadata in metadata:
        queue.enqueue(
            lambda: transcribe_file_and_extract_speakers(
                str(file_metadata.id), db, audio_editor
            )
        )
    return "Files and transcriptions are being processed in the background."


def save_schema(schema_name: str, schema_fields: str):
    unvalidated_schema = UnvalidatedSchema(
        name=schema_name, fields=schema_fields.split(",")
    )
    create_and_save_schema(unvalidated_schema, db, ai_model_client)
    return "Schema saved successfully."


def extract_info(user_ids: list[str]):
    transcriptions = db.get_transcriptions_by_user_ids(user_ids)
    schema = db.get_latest_schema()
    csv_bytes = extract_info_from_transcriptions(
        schema, transcriptions, db, ai_model_client, "output.csv"
    )
    return "Information extracted. Download the CSV file.", gr.File.update(
        value=csv_bytes, label="Download CSV"
    )


with gr.Blocks() as app:
    gr.Markdown("## File Upload and Transcription")
    user_id_input = gr.Textbox(label="User ID")
    file_input = gr.File(label="Files", file_count="multiple")
    save_button = gr.Button("Save Files and Transcriptions")
    save_output = gr.Textbox(label="Output")
    save_button.click(
        save_files_and_transcriptions,
        inputs=[user_id_input, file_input],
        outputs=save_output,
    )

    gr.Markdown("## Schema Definition")
    schema_name_input = gr.Textbox(label="Schema Name")
    schema_fields_input = gr.Textbox(label="Schema Fields (comma-separated)")
    schema_save_button = gr.Button("Save Schema")
    schema_save_output = gr.Textbox(label="Output")
    schema_save_button.click(
        save_schema,
        inputs=[schema_name_input, schema_fields_input],
        outputs=schema_save_output,
    )

    gr.Markdown("## Information Extraction")
    user_ids_input = gr.CheckboxGroup(label="User IDs", choices=db.get_all_user_ids())
    extract_button = gr.Button("Extract Information")
    extract_output = gr.Textbox(label="Output")
    csv_download = gr.File(label="Download CSV", interactive=False)
    extract_button.click(
        extract_info, inputs=user_ids_input, outputs=[extract_output, csv_download]
    )

queue.start_workers(num_workers=4)  # Start worker threads
app.launch()

In [8]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore